# Analisis Exploratorio de Datos (EDA)
## Produccion Agricola del Peru - MIDAGRI/SIEA

Este notebook sigue la metodologia CRISP-DM para la fase de **comprension de datos**.
El objetivo es explorar los datos de produccion agricola para entender su estructura,
distribuciones, relaciones y anomalias antes de construir el pipeline ETL.

### Contenido
1. Carga y estructura del dataset
2. Estadisticas descriptivas (tendencia central y dispersion)
3. Distribucion de variables numericas
4. Analisis por departamento y region natural
5. Series temporales y estacionalidad
6. Correlaciones entre variables
7. Deteccion de outliers (metodo IQR)
8. Hallazgos y recomendaciones para el pipeline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

DATA_DIR = Path('../data/raw')
print(f'Archivos disponibles: {list(DATA_DIR.glob("*"))}')

## 1. Carga y estructura del dataset

Primero cargamos los datos crudos descargados del SIEA/MIDAGRI y revisamos
la estructura basica: tipos de datos, dimensiones, primeras filas.

In [ ]:
# Cargar el dataset (ajustar el nombre del archivo segun lo descargado)
# df = pd.read_csv(DATA_DIR / 'produccion_agricola.csv', encoding='latin-1')

# Para este ejemplo usamos los datos de prueba
df = pd.read_csv('../tests/fixtures/sample_data.csv')

print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Memoria: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print()
print('Tipos de datos:')
print(df.dtypes)
print()
df.head(10)

In [ ]:
# Valores nulos por columna
nulls = df.isnull().sum()
nulls_pct = (df.isnull().mean() * 100).round(2)
pd.DataFrame({'nulos': nulls, 'porcentaje': nulls_pct}).query('nulos > 0')

## 2. Estadisticas descriptivas

Las medidas de tendencia central (media, mediana) y dispersion (desviacion estandar,
rango intercuartilico) nos dan una primera vision de como se distribuyen los datos.
El coeficiente de variacion (CV) indica la dispersion relativa: un CV > 100% sugiere
alta heterogeneidad en los datos.

In [ ]:
numeric_cols = ['superficie_sembrada_ha', 'superficie_cosechada_ha',
                'produccion_toneladas', 'rendimiento_kg_ha', 'precio_chacra_soles']

stats = df[numeric_cols].describe().T
stats['mediana'] = df[numeric_cols].median()
stats['cv_%'] = (df[numeric_cols].std() / df[numeric_cols].mean() * 100).round(1)
stats['asimetria'] = df[numeric_cols].skew().round(2)
stats['curtosis'] = df[numeric_cols].kurtosis().round(2)
stats

## 3. Distribucion de variables numericas

Los histogramas revelan si la distribucion es normal, sesgada o multimodal.
En datos agricolas, la produccion suele tener sesgo positivo (pocos departamentos
concentran la mayor produccion). Los box plots complementan mostrando outliers.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    df[col].hist(bins=20, ax=ax, edgecolor='black', alpha=0.7, color='#2E86AB')
    ax.axvline(df[col].mean(), color='#E84855', linestyle='--', label=f'Media: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='#44BBA4', linestyle='-', label=f'Mediana: {df[col].median():.1f}')
    ax.set_title(col.replace('_', ' ').title(), fontsize=10)
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Distribucion de variables numericas', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/quality/distribucion_variables.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(16, 5))

for i, col in enumerate(numeric_cols):
    bp = axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                         boxprops=dict(facecolor='#2E86AB', alpha=0.6))
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=9)

plt.suptitle('Box plots - Identificacion de outliers', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/quality/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Analisis por departamento y region natural

Peru tiene 3 regiones naturales (Costa, Sierra, Selva) con caracteristicas
agricolas muy diferentes. La Costa concentra la agricultura de exportacion,
la Sierra los cultivos tradicionales y la Selva los cultivos tropicales.

In [ ]:
# Produccion total por departamento
prod_dept = df.groupby('departamento')['produccion_toneladas'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
prod_dept.plot(kind='barh', ax=ax, color='#2E86AB', edgecolor='black', alpha=0.8)
ax.set_xlabel('Produccion total (toneladas)')
ax.set_title('Produccion agricola por departamento')
plt.tight_layout()
plt.savefig('../data/quality/produccion_por_departamento.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top cultivos por produccion
prod_cultivo = df.groupby('cultivo').agg({
    'produccion_toneladas': 'sum',
    'superficie_cosechada_ha': 'sum',
    'precio_chacra_soles': 'mean',
}).round(2)

prod_cultivo['rendimiento_efectivo'] = (
    prod_cultivo['produccion_toneladas'] / prod_cultivo['superficie_cosechada_ha'] * 1000
).round(0)

prod_cultivo.sort_values('produccion_toneladas', ascending=False)

## 5. Series temporales y estacionalidad

La agricultura peruana tiene estacionalidad marcada por las campanas agricolas
(agosto a julio del siguiente ano). Los meses de siembra son agosto-enero
y los de cosecha marzo-julio.

In [ ]:
# Produccion por mes
nombres_mes = {1: 'Ene', 2: 'Feb', 3: 'Mar', 4: 'Abr', 5: 'May', 6: 'Jun',
               7: 'Jul', 8: 'Ago', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dic'}

prod_mensual = df.groupby('mes')['produccion_toneladas'].sum()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(prod_mensual.index, prod_mensual.values, color='#2E86AB', edgecolor='black', alpha=0.8)

# Marcar epocas de siembra (ago-ene) y cosecha (mar-jul)
for i, bar in enumerate(bars):
    mes = prod_mensual.index[i]
    if mes in [3, 4, 5, 6, 7]:
        bar.set_color('#44BBA4')  # cosecha
    elif mes in [8, 9, 10, 11, 12, 1]:
        bar.set_color('#E8D44D')  # siembra

ax.set_xticks(range(1, 13))
ax.set_xticklabels([nombres_mes[m] for m in range(1, 13)])
ax.set_ylabel('Produccion (toneladas)')
ax.set_title('Produccion por mes - Estacionalidad agricola')
ax.legend(['Cosecha (Mar-Jul)', 'Siembra (Ago-Ene)'], loc='upper right')
plt.tight_layout()
plt.savefig('../data/quality/estacionalidad.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlaciones entre variables

La matriz de correlacion de Pearson mide la relacion lineal entre variables.
Esperamos una correlacion fuerte entre superficie cosechada y produccion
(a mayor area, mayor produccion) y una relacion inversa entre rendimiento
y precio en chacra (cultivos de alto rendimiento tienden a tener menor precio).

In [ ]:
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            xticklabels=[c.replace('_', '\n') for c in numeric_cols],
            yticklabels=[c.replace('_', '\n') for c in numeric_cols])
ax.set_title('Matriz de correlacion de Pearson')
plt.tight_layout()
plt.savefig('../data/quality/correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Deteccion de outliers (metodo IQR)

El metodo del rango intercuartilico (IQR) identifica valores atipicos como
aquellos que caen fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]. En datos agricolas,
los outliers pueden ser datos reales (un departamento con produccion excepcional)
o errores de captura.

In [ ]:
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    if len(outliers) > 0:
        print(f'{col}: {len(outliers)} outliers detectados')
        print(f'  Rango normal: [{lower:.1f}, {upper:.1f}]')
        print(f'  Valores atipicos: {outliers[col].values}')
        print()

## 8. Hallazgos y recomendaciones

### Hallazgos del EDA
- Documentar aqui los patrones encontrados en los datos reales
- Relaciones entre variables
- Anomalias detectadas
- Cobertura temporal y geografica

### Recomendaciones para el pipeline
- Definir rangos de validacion basados en los percentiles observados
- Implementar tratamiento especifico para outliers por cultivo
- Considerar imputacion de valores nulos segun el patron de missing data
- Agregar validacion de consistencia: produccion <= superficie * rendimiento_maximo_historico